# Notebook pour faire l'entraînement sur tous les matériaux

NB: S'il y a des erreurs lors de l'éxécution, il faut soit ajouter un "." devant le mot "utils" à la ligne 3 du fichier database.py (situé dans le dossier merlDB), soit le retirer.

In [1]:
import torch
import torch.utils.data as torchdata
import torchsummary
import neural_data as nd
import merlDB.database as db
import argparse
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import mae_model_brdf

In [ ]:
# Si c'est vide ça veut dire qu'on fait l'entraînement sur tous les matériaux, sinon remplacer par un matériaux
materiaux= ""

# Pourcentage de triplets (R,G,B) masqués (recommandé 0.5)
ratio_masking = 0.5

# Quantité de données à utiliser pour l'entraînement (min 1 max 100)
train_size = 100

# Batch size
batch_size = 20

# Epochs
epochs = 200

# poids ajouté (valeur par défaut = 1)
poids = 2 

# taille de l'espace latent (valeur par défaut = 768, il faut que ce soit divisible par le nombre de tête d'attention (par défaut 12))
dim_latent = 768

# norme de la fontion de perte (par défaut 2, sinon mettre 1 pour une loss L1, autre chose va faire crash le programme)
norme = 2

## Chargement des données MERL

In [6]:
def get_args_parser():
    parser = argparse.ArgumentParser('MAE pre-training', add_help=False)
    parser.add_argument('--batch_size', default=64, type=int,
                        help='Batch size per GPU (effective batch size is batch_size * accum_iter * # gpus')
    parser.add_argument('--epochs', default=400, type=int)
    parser.add_argument('--accum_iter', default=1, type=int,
                        help='Accumulate gradient iterations (for increasing the effective batch size under memory constraints)')

    # Model parameters
    parser.add_argument('--model', default='mae_brdf', type=str, metavar='MODEL',
                        help='Name of model to train')

    parser.add_argument('--input_size', default=(90, 90, 180), type=int,
                        help='images input size')

    parser.add_argument('--mask_ratio', default=ratio_masking, type=float,
                        help='Masking ratio (percentage of removed patches).')

    parser.add_argument('--norm_pix_loss', action='store_true',
                        help='Use (per-patch) normalized pixels as targets for computing loss')
    parser.set_defaults(norm_pix_loss=False)

    # Optimizer parameters
    parser.add_argument('--weight_decay', type=float, default=0.05,
                        help='weight decay (default: 0.05)')

    parser.add_argument('--lr', type=float, default=1e-4, metavar='LR',
                        help='learning rate (absolute lr)')
    parser.add_argument('--blr', type=float, default=1e-3, metavar='LR',
                        help='base learning rate: absolute_lr = base_lr * total_batch_size / 256')
    parser.add_argument('--min_lr', type=float, default=0., metavar='LR',
                        help='lower lr bound for cyclic schedulers that hit 0')

    parser.add_argument('--warmup_epochs', type=int, default=40, metavar='N',
                        help='epochs to warmup LR')

    # Dataset parameters /!\ à modifier quand on aura le dataset
    parser.add_argument('--data_path', default='/datasets01/imagenet_full_size/061417/', type=str,
                        help='dataset path')

    parser.add_argument('--output_dir', default='./output_dir',
                        help='path where to save, empty for no saving')
    parser.add_argument('--log_dir', default='./output_dir',
                        help='path where to tensorboard log')
    parser.add_argument('--device', default='cuda',
                        help='device to use for training / testing')
    parser.add_argument('--seed', default=0, type=int)
    parser.add_argument('--resume', default='',
                        help='resume from checkpoint')

    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    parser.add_argument('--num_workers', default=10, type=int)
    parser.add_argument('--pin_mem', action='store_true',
                        help='Pin CPU memory in DataLoader for more efficient (sometimes) transfer to GPU.')
    parser.add_argument('--no_pin_mem', action='store_false', dest='pin_mem')
    parser.set_defaults(pin_mem=True)

    # distributed training parameters
    parser.add_argument('--world_size', default=1, type=int,
                        help='number of distributed processes')
    parser.add_argument('--local_rank', default=-1, type=int)
    parser.add_argument('--dist_on_itp', action='store_true')
    parser.add_argument('--dist_url', default='env://',
                        help='url used to set up distributed training')

    return parser


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device", device)

parser = argparse.ArgumentParser()
parser.add_argument('--outdir', default='results/')
parser.add_argument('--merldir', default='merlDB/db/brdfs/')
parser.add_argument('--mediandir', default='merlDB/db/')
parser.add_argument('--batch_size', default=batch_size)
parser.add_argument('--train_size', default=train_size)
parser.add_argument('--monomat', default=None)
parser.add_argument('--epochs', default=epochs)
parser.add_argument('--lr', default=1e-4)
parser.add_argument('--load_previous', default=False)
parser.add_argument('--materiau', default=materiaux) 
args, unknown = parser.parse_known_args()

medians = db.readbin(args.mediandir + 'merl_median.binary')

def median_mapping(albedos, epsilon=0.002):
    return np.log((albedos + epsilon)/(medians + epsilon) + 1)

print(dim_latent)
assert dim_latent % 2 == 0
mae = mae_model_brdf.MaskedAutoencoderViT3D(poids=poids, embed_dim=dim_latent, norme=norme)

dbuilder = db.DBuilder(db_path=args.merldir, albedo_mapping=median_mapping)
test_dbuilder = db.DBuilder(db_path=args.merldir, albedo_mapping=median_mapping)
ldb = dbuilder.list_db()
print(ldb)
if args.load_previous:
    print('LOADING MODEL')
    train_info = np.load(args.outdir + 'train_info.npy', allow_pickle=True).item()
    mats = train_info.get('train_mats')
else:
    if args.materiau != "" :
        mats = [args.materiau]
        test_mats = [args.materiau]
    else :
        mats_idx = np.random.choice(range(len(ldb)), int(args.train_size), replace=False)
        mats = [ldb[idx] for idx in mats_idx]
        test_mats = [mat for mat in ldb if mat not in mats]

if not args.monomat is None:
    print("problème")
    mats = [args.monomat]

print(mats)
for mat in tqdm(mats, 'Loading mats'):
    if mat != "Rea" :
        dbuilder.load_mat(mat)

if torch.cuda.is_available():
    for mat in tqdm(test_mats, 'Loading test mats'):
        if mat != "Rea" :
            test_dbuilder.load_mat(mat)

dataset = nd.DBridge(dbuilder, device)
test_dataset = nd.DBridge(test_dbuilder, device)
del dbuilder
del test_dbuilder

"""
if not torch.cuda.is_available():
    merl_ae = nd.MockMerlAE()
else:
    merl_ae = nd.MerlAutoEncoder()
    if args.load_previous:
        merl_ae.load_state_dict(torch.load(args.outdir + 'melr_ae.pt'))
"""

if args.load_previous:
        mae.load_state_dict(torch.load(args.outdir + 'mae.pt'))

#trainer = nd.MerlAETrainer(device, dataset, mae=merl_ae, lr=float(args.lr), batch_size=int(args.batch_size), shuffle=True)
args2 = get_args_parser()
args2, unknown = args2.parse_known_args()
trainer = nd.MAETrainer(device, dataset, mae=mae, lr=float(args.lr), batch_size=int(args.batch_size), shuffle=True, args=args2)

torchsummary.summary(trainer.mae, nd.INPUT_SHAPE)


Using device cuda
768
['alum-bronze', 'alumina-oxide', 'aluminium', 'aventurnine', 'beige-fabric', 'black-fabric', 'black-obsidian', 'black-oxidized-steel', 'black-phenolic', 'black-soft-plastic', 'blue-acrylic', 'blue-fabric', 'blue-metallic-paint', 'blue-metallic-paint2', 'blue-rubber', 'brass', 'cherry-235', 'chrome-steel', 'chrome', 'colonial-maple-223', 'color-changing-paint1', 'color-changing-paint2', 'color-changing-paint3', 'dark-blue-paint', 'dark-red-paint', 'dark-specular-fabric', 'delrin', 'fruitwood-241', 'gold-metallic-paint', 'gold-metallic-paint2', 'gold-metallic-paint3', 'gold-paint', 'gray-plastic', 'grease-covered-steel', 'green-acrylic', 'green-fabric', 'green-latex', 'green-metallic-paint', 'green-metallic-paint2', 'green-plastic', 'hematite', 'ipswich-pine-221', 'light-brown-fabric', 'light-red-paint', 'maroon-plastic', 'natural-209', 'neoprene-rubber', 'nickel', 'nylon', 'orange-paint', 'pearl-paint', 'pickled-oak-260', 'pink-fabric', 'pink-fabric2', 'pink-felt',

Loading mats:   1%|          | 1/100 [00:00<00:16,  5.93it/s]

merlDB/db/brdfs/alumina-oxide.binary
merlDB/db/brdfs/white-paint.binary


Loading mats:   3%|▎         | 3/100 [00:00<00:15,  6.15it/s]

merlDB/db/brdfs/pvc.binary
merlDB/db/brdfs/two-layer-gold.binary


...

merlDB/db/brdfs/red-specular-plastic.binary
merlDB/db/brdfs/violet-rubber.binary


Loading mats:  99%|█████████▉| 99/100 [00:15<00:00,  5.81it/s]

merlDB/db/brdfs/pink-jasper.binary
merlDB/db/brdfs/special-walnut-224.binary


Loading test mats: 100%|██████████| 1/1 [00:00<00:00,  5.99it/s]


merlDB/db/brdfs/gold-paint.binary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1        [-1, 768, 6, 6, 12]       7,776,768
      PatchEmbed3D-2             [-1, 432, 768]               0
         LayerNorm-3             [-1, 109, 768]           1,536
            Linear-4            [-1, 109, 2304]       1,771,776
          Identity-5          [-1, 12, 109, 64]               0
          Identity-6          [-1, 12, 109, 64]               0
          Identity-7             [-1, 109, 768]               0
            Linear-8             [-1, 109, 768]         590,592
           Dropout-9             [-1, 109, 768]               0
        Attention-10             [-1, 109, 768]               0
         Identity-11             [-1, 109, 768]               0
         Identity-12             [-1, 109, 768]               0
        LayerNorm-13             [-1, 109, 768]           1,536
     

## Lancement de l'entraînement

In [7]:
for epoch in range(int(args.epochs)):
    print('EPOCH',epoch)
    trainer.train_epoch()
    total_preds = 0
    total_correct_preds = 0
    for i,data in enumerate(torchdata.DataLoader(test_dataset)):
        X = data['values']
        _, out, _ , latent = trainer.mae(X)
      

EPOCH 0
Epoch: [0]  [0/5]  eta: 0:03:03  lr: 0.000000  loss: 1.5500 (1.5500)  time: 36.7020  data: 0.3082  max mem: 11847
Epoch: [0]  [4/5]  eta: 0:00:39  lr: 0.000002  loss: 1.4614 (1.4435)  time: 39.7322  data: 0.2459  max mem: 12814
Epoch: [0] Total time: 0:03:18 (39.7325 s / it)
Averaged stats: lr: 0.000002  loss: 1.4614 (1.4435)
loss 1.373008370399475
EPOCH 1
Epoch: [1]  [0/5]  eta: 0:02:23  lr: 0.000003  loss: 1.3671 (1.3671)  time: 28.7328  data: 0.4527  max mem: 12814
Epoch: [1]  [4/5]  eta: 0:00:34  lr: 0.000005  loss: 1.4074 (1.4191)  time: 34.7324  data: 0.1308  max mem: 12814
Epoch: [1] Total time: 0:02:53 (34.7326 s / it)
Averaged stats: lr: 0.000005  loss: 1.4074 (1.4191)
loss 1.4474438428878784
EPOCH 2


KeyboardInterrupt: 

## Sauvegarde du modèle

In [ ]:
def save_model():
    if torch.cuda.is_available() or True:
        if args.load_previous:
            train_info = np.load(args.outdir + 'train_info.npy', allow_pickle=True).item()
            train_info['losses'] = train_info['losses'] + trainer.losses
        else:
            train_info = {'losses' : trainer.losses, 'train_mats': mats}
        np.save(args.outdir + 'train_info.npy', train_info, allow_pickle=True)
        fig = plt.figure()
        ax = fig.add_subplot(111)
        all_losses = train_info['losses']
        ax.plot(range(len(all_losses)), all_losses)
        fig.savefig(args.outdir + 'losses.png')
        plt.show()
        plt.close(fig)
        torch.save(trainer.mae.state_dict(), args.outdir + 'mae.pt')


In [ ]:
save_model()

# Comparaison des rendus entre la vérité et la prédiction

### Récupération du modèle à partir des poids enregistrés

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mae = mae_model_brdf.MaskedAutoencoderViT3D(poids=2, embed_dim=dim_latent)
mae.load_state_dict(torch.load("./results/mae.pt"))
mae.to(device)
mae.eval()

FileNotFoundError: [Errno 2] No such file or directory: './results/mae.pt'

### Exemple de génération des rendus avec un matériau

In [ ]:
materiaux_res="beige-fabric"

In [5]:
medians = db.readbin('./merlDB/db/merl_median.binary')

def median_mapping(albedos, epsilon=0.002):
    return np.log((albedos + epsilon)/(medians + epsilon) + 1)

def median_unmapping(mapped_albedos, epsilon=0.002):
    return (np.exp(mapped_albedos) -1) * (medians + epsilon)  - epsilon

In [ ]:
from merlDB.database import writebin, readbin
materiau = f"./merlDB/db/brdfs/{materiaux_res}.binary"
valeur = torch.Tensor(readbin(materiau))
materiau_pred = f"./results/prediction/{materiaux_res}-pred.binary"
valeur_pred = torch.Tensor(readbin(materiau_pred))
print(torch.sum(valeur == valeur_pred))
print(valeur[3,78,26,:])
print(valeur_pred[3,78,26,:])

save_dbuilder = db.DBuilder(db_path=args.merldir, albedo_mapping=median_mapping)
save_dbuilder.load_mat(materiaux_res)
save_dataset = nd.DBridge(save_dbuilder, device)
del save_dbuilder
for i,data in enumerate(torchdata.DataLoader(save_dataset)):
        X = data['values']
        _, prediction, _ , latent = mae(X)
        prediction = mae.unpatchify(prediction)
        Y = torch.swapaxes(prediction.squeeze(), 0, -1).detach().cpu().numpy()
        Y = median_unmapping(Y)
        writebin(f"./results/prediction/{materiaux_res}-pred.binary", Y)

tensor(0)
tensor([1.9121, 1.5190, 1.3843])
tensor([1.6890, 1.6271, 1.5869])
merlDB/db/brdfs/alum-bronze.binary


In [ ]:
!python3 ./merlDB/rendering.py --mat "{materiaux_res}"
!python3 ./merlDB/rendering.py --mat "{materiaux_res}-pred" --merldir "./results/prediction/"

## Génération des predictions pour tous les matériaux

In [9]:
materiaux = ['aventurnine',
 'ipswich-pine-221',
 'pure-rubber',
 'gold-metallic-paint',
 'blue-metallic-paint2',
 'gold-metallic-paint3',
 'maroon-plastic',
 'red-metallic-paint',
 'pink-fabric',
 'pink-jasper',
 'red-specular-plastic',
 'green-metallic-paint',
 'beige-fabric',
 'neoprene-rubber',
 'color-changing-paint1',
 'specular-violet-phenolic',
 'purple-paint',
 'dark-specular-fabric',
 'gray-plastic',
 'blue-metallic-paint',
 'hematite',
 'yellow-matte-plastic',
 'silver-paint',
 'dark-red-paint',
 'black-oxidized-steel',
 'green-fabric',
 'alumina-oxide',
 'white-paint',
 'yellow-paint',
 'blue-fabric',
 'chrome',
 'light-brown-fabric',
 'red-fabric2',
 'brass',
 'white-diffuse-bball',
 'yellow-plastic',
 'specular-red-phenolic',
 'grease-covered-steel',
 'delrin',
 'white-fabric',
 'gold-paint',
 'pearl-paint',
 'special-walnut-224',
 'cherry-235',
 'green-metallic-paint2',
 'silicon-nitrade',
 'colonial-maple-223',
 'specular-black-phenolic',
 'black-phenolic',
 'violet-rubber',
 'pink-fabric2',
 'specular-white-phenolic',
 'light-red-paint',
 'fruitwood-241',
 'specular-blue-phenolic',
 'gold-metallic-paint2',
 'teflon',
 'silver-metallic-paint2',
 'pvc',
 'green-latex',
 'color-changing-paint2',
 'polyethylene',
 'pink-felt',
 'specular-yellow-phenolic',
 'ss440',
 'white-fabric2',
 'black-obsidian',
 'blue-acrylic',
 'violet-acrylic',
 'chrome-steel',
 'polyurethane-foam',
 'nylon',
 'color-changing-paint3',
 'specular-orange-phenolic',
 'dark-blue-paint',
 'white-acrylic',
 'alum-bronze',
 'red-phenolic',
 'specular-maroon-phenolic',
 'white-marble',
 'two-layer-gold',
 'steel',
 'red-fabric',
 'black-soft-plastic',
 'natural-209',
 'red-plastic',
 'pink-plastic',
 'nickel',
 'two-layer-silver',
 'yellow-phenolic',
 'black-fabric',
 'aluminium',
 'pickled-oak-260',
 'green-acrylic',
 'silver-metallic-paint',
 'green-plastic',
 'tungsten-carbide',
 'blue-rubber',
 'orange-paint',
 'specular-green-phenolic']



In [ ]:
for materiaux_res in materiaux:
    !python3 ./merlDB/rendering.py --mat "{materiaux_res}"
    save_dbuilder = db.DBuilder(db_path=args.merldir, albedo_mapping=median_mapping)
    save_dbuilder.load_mat(materiaux_res)
    save_dataset = nd.DBridge(save_dbuilder, device)
    del save_dbuilder
    for i,data in enumerate(torchdata.DataLoader(save_dataset)):
            X = data['values']
            _, prediction, _ , latent = trainer.mae(X)
            prediction = trainer.mae.unpatchify(prediction)
            Y = torch.swapaxes(prediction.squeeze(), 0, -1).detach().cpu().numpy()
            Y = median_unmapping(Y)
            writebin(f"./results/prediction/{materiaux_res}-pred.binary", Y)
    !python3 ./merlDB/rendering.py --mat "{materiaux_res}-pred" --merldir "./results/prediction/"

merlDB/db/brdfs/aventurnine.binary
['dark-red-paint-pred', 'pure-rubber-pred', 'blue-acrylic-pred', 'specular-red-phenolic-pred', 'green-metallic-paint2-pred', 'color-changing-paint1-pred', 'silver-paint-pred', 'red-fabric2-pred', 'colonial-maple-223-pred', 'pvc-pred', 'ss440-pred', 'specular-yellow-phenolic-pred', 'red-phenolic-pred', 'specular-black-phenolic-pred', 'red-fabric-pred', 'light-brown-fabric-pred', 'gold-metallic-paint2-pred', 'white-fabric2-pred', 'delrin-pred', 'cherry-235-pred', 'gold-paint-pred', 'two-layer-gold-pred', 'specular-maroon-phenolic-pred', 'grease-covered-steel-pred', 'specular-green-phenolic-pred', 'gold-metallic-paint3-pred', 'pink-fabric-pred', 'special-walnut-224-pred', 'aluminium-pred', 'white-marble-pred', 'natural-209-pred', 'specular-violet-phenolic-pred', 'green-plastic-pred', 'neoprene-rubber-pred', 'specular-white-phenolic-pred', 'black-phenolic-pred', 'teflon-pred', 'pink-plastic-pred', 'blue-rubber-pred', 'color-changing-paint3-pred', 'pink-fa